# Bias in Bios Concept-QA

Train the reusable Concept-QA module for `LabHC/bias_in_bios`.

Main choices:

- input encoder: `sentence-transformers/all-MiniLM-L6-v2`
- query vocabulary: fixed BIOS concept set
- supervision: calibrated encoder-similarity soft targets
- artifact: Concept-QA checkpoint and split-level cache

In [ ]:
# %pip install -e ..
# %pip install datasets sentence-transformers scikit-learn -q

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from torch.utils.data import DataLoader, TensorDataset

from staq.config import default_paths, detect_device
from staq.models import ConceptNet2
from staq.training import seed_everything

In [3]:
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
paths = default_paths(repo_root)
paths.ensure_artifact_dirs()

device = detect_device()
random_seed = 13
seed_everything(random_seed)

encoder_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_dim = 384

dataset_name = "LabHC/bias_in_bios"
text_column = "hard_text"
target_column = "profession"
sensitive_column = "gender"

profession_names = [
    "accountant",
    "architect",
    "attorney",
    "chiropractor",
    "comedian",
    "composer",
    "dentist",
    "dietitian",
    "dj",
    "filmmaker",
    "interior_designer",
    "journalist",
    "model",
    "nurse",
    "painter",
    "paralegal",
    "pastor",
    "personal_trainer",
    "photographer",
    "physician",
    "poet",
    "professor",
    "psychologist",
    "rapper",
    "software_engineer",
    "surgeon",
    "teacher",
    "yoga_teacher",
]

gender_names = ["male", "female"]

{
    "device": str(device),
    "encoder_name": encoder_name,
    "embedding_dim": embedding_dim,
    "dataset_name": dataset_name,
}

{'device': 'cuda',
 'encoder_name': 'sentence-transformers/all-MiniLM-L6-v2',
 'embedding_dim': 384,
 'dataset_name': 'LabHC/bias_in_bios'}

In [4]:
raw_train = load_dataset(dataset_name, split="train")
raw_dev = load_dataset(dataset_name, split="dev")
raw_test = load_dataset(dataset_name, split="test")

{
    "train": len(raw_train),
    "dev": len(raw_dev),
    "test": len(raw_test),
    "columns": raw_train.column_names,
}

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-0ab65b32c47407(…):   0%|          | 0.00/64.9M [00:00<?, ?B/s]

data/test-00000-of-00001-5598c840ce8de1e(…):   0%|          | 0.00/24.9M [00:00<?, ?B/s]

data/dev-00000-of-00001-e6551072fff26949(…):   0%|          | 0.00/9.95M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/257478 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/99069 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/39642 [00:00<?, ? examples/s]

{'train': 257478,
 'dev': 39642,
 'test': 99069,
 'columns': ['hard_text', 'profession', 'gender']}

In [5]:
concepts_path = repo_root / "assets" / "concepts" / "bias_in_bios.csv"
concepts_df = pd.read_csv(concepts_path)

concept_names = concepts_df["concept"].tolist()
concept_texts = concepts_df["prompt"].tolist()
sensitive_concepts = concepts_df.query("kind != 'utility'")["concept"].tolist()
sensitive_mask = torch.tensor(concepts_df["kind"].ne("utility").to_numpy(), dtype=torch.bool)

{
    "num_concepts": len(concepts_df),
    "by_kind": concepts_df["kind"].value_counts().to_dict(),
    "num_sensitive_concepts": int(sensitive_mask.sum()),
}

{'num_concepts': 40,
 'by_kind': {'utility': 33, 'proxy_sensitive': 5, 'direct_sensitive': 2},
 'num_sensitive_concepts': 7}

In [6]:
encoder = SentenceTransformer(encoder_name, device=str(device))

concept_embeddings = encoder.encode(
    concept_texts,
    batch_size=64,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).to(device)

concept_dictionary = concept_embeddings.T.contiguous()

{
    "concept_embeddings": tuple(concept_embeddings.shape),
    "concept_dictionary": tuple(concept_dictionary.shape),
}

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

{'concept_embeddings': (40, 384), 'concept_dictionary': (384, 40)}

In [7]:
sample_rows = raw_train.select(range(8))
sample_texts = [row[text_column] for row in sample_rows]

sample_embeddings = encoder.encode(
    sample_texts,
    batch_size=8,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=False,
).to(device)

sample_concept_qa_inputs = torch.cat(
    [
        sample_embeddings.unsqueeze(1).repeat(1, len(concept_names), 1),
        concept_embeddings.unsqueeze(0).repeat(len(sample_texts), 1, 1),
    ],
    dim=2,
).reshape(len(sample_texts) * len(concept_names), -1)

{
    "sample_embeddings": tuple(sample_embeddings.shape),
    "concept_qa_input_shape": tuple(sample_concept_qa_inputs.shape),
    "expected_concept_qa_input_dim": embedding_dim * 2,
}

{'sample_embeddings': (8, 384),
 'concept_qa_input_shape': (320, 768),
 'expected_concept_qa_input_dim': 768}

In [8]:
pd.DataFrame(
    {
        "profession": [profession_names[row[target_column]] for row in sample_rows],
        "gender": [gender_names[row[sensitive_column]] for row in sample_rows],
        "text": sample_texts,
    }
)

,profession,gender,text
0,professor,male,He is also the project lead of and major contr...
1,nurse,female,"She is able to assess, diagnose and treat mino..."
2,attorney,female,"Prior to law school, Brittni graduated magna c..."
3,journalist,male,He regularly contributes to India’s First Onli...
4,professor,male,He completed his medical degree at Northwester...
5,attorney,male,Steve has practiced law in Kentucky for over 3...
6,professor,male,"His research is in information archiving, anal..."
7,poet,female,"Trained as a doctor, she lives in Algiers wher..."


In [9]:
encoding_batch_size = 256
raw_splits = {
    "train": raw_train,
    "dev": raw_dev,
    "test": raw_test,
}


def encode_bios_split(split):
    texts = [row[text_column] for row in split]
    embeddings = encoder.encode(
        texts,
        batch_size=encoding_batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).to(device)
    professions = [profession_names[row[target_column]] for row in split]
    genders = [gender_names[row[sensitive_column]] for row in split]
    return {
        "texts": texts,
        "embeddings": embeddings,
        "similarity_scores": embeddings @ concept_embeddings.T,
        "profession_labels": professions,
        "gender_labels": genders,
        "profession_ids": torch.tensor([row[target_column] for row in split], dtype=torch.long),
        "gender_ids": torch.tensor([row[sensitive_column] for row in split], dtype=torch.long),
    }


encoded_splits = {
    split_name: encode_bios_split(split)
    for split_name, split in raw_splits.items()
}

{
    split_name: {
        "embeddings": tuple(split_data["embeddings"].shape),
        "similarity_scores": tuple(split_data["similarity_scores"].shape),
    }
    for split_name, split_data in encoded_splits.items()
}

Batches:   0%|          | 0/1006 [00:00<?, ?it/s]

Batches:   0%|          | 0/155 [00:00<?, ?it/s]

Batches:   0%|          | 0/387 [00:00<?, ?it/s]

{'train': {'embeddings': (257478, 384), 'similarity_scores': (257478, 40)},
 'dev': {'embeddings': (39642, 384), 'similarity_scores': (39642, 40)},
 'test': {'embeddings': (99069, 384), 'similarity_scores': (99069, 40)}}

In [10]:
train_similarity_scores = encoded_splits["train"]["similarity_scores"]
concept_medians = train_similarity_scores.median(dim=0).values
concept_stds = train_similarity_scores.std(dim=0).clamp_min(1e-6)
calibration_temperature = 1.5


def calibrate_similarity_scores(similarity_scores):
    return torch.sigmoid(
        (similarity_scores - concept_medians.unsqueeze(0))
        / (calibration_temperature * concept_stds.unsqueeze(0))
    )


for split_data in encoded_splits.values():
    split_data["calibrated_soft_targets"] = calibrate_similarity_scores(split_data["similarity_scores"])

train_targets = encoded_splits["train"]["calibrated_soft_targets"]
calibrated_summary = pd.DataFrame(
    {
        "concept": concept_names,
        "kind": concepts_df["kind"],
        "mean": train_targets.mean(dim=0).cpu().numpy(),
        "std": train_targets.std(dim=0).cpu().numpy(),
        "p05": torch.quantile(train_targets, 0.05, dim=0).cpu().numpy(),
        "p50": torch.quantile(train_targets, 0.50, dim=0).cpu().numpy(),
        "p95": torch.quantile(train_targets, 0.95, dim=0).cpu().numpy(),
    }
)

calibrated_summary.sort_values("mean", ascending=False)

,concept,kind,mean,std,p05,p50,p95
2,clinical_care,utility,0.536263,0.150468,0.333855,0.500000,0.807961
18,visual_art_practice,utility,0.524735,0.144932,0.327658,0.500000,0.820223
9,manual_body_treatment,utility,0.524383,0.148493,0.314861,0.500001,0.794846
22,literary_writing,utility,0.518889,0.149954,0.303046,0.500000,0.791995
19,camera_or_photography_work,utility,0.518163,0.144470,0.313244,0.500002,0.813968
17,interior_or_decorative_design,utility,0.518150,0.146530,0.307515,0.500001,0.793418
4,surgical_work,utility,0.517680,0.148738,0.299947,0.500000,0.790008
25,comedy_or_stage_performance,utility,0.516448,0.149445,0.292292,0.500001,0.785051
15,technical_infrastructure,utility,0.516092,0.144415,0.305296,0.500001,0.795259
5,nursing_care,utility,0.515206,0.150992,0.288067,0.500003,0.779323


In [11]:
def show_top_calibrated_concepts(row_index, split_name="train", top_k=8):
    split_data = encoded_splits[split_name]
    scores = split_data["calibrated_soft_targets"][row_index].detach().cpu().numpy()
    top_indices = np.argsort(scores)[::-1][:top_k]
    print(f"profession: {split_data['profession_labels'][row_index]}")
    print(f"gender: {split_data['gender_labels'][row_index]}")
    print(split_data["texts"][row_index])
    return pd.DataFrame(
        {
            "concept": [concept_names[idx] for idx in top_indices],
            "kind": [concepts_df.loc[idx, "kind"] for idx in top_indices],
            "score": scores[top_indices],
        }
    )


show_top_calibrated_concepts(0)

profession: professor
gender: male
He is also the project lead of and major contributor to the open source assembler/simulator "EASy68K." He earned a master’s degree in computer science from the University of Michigan-Dearborn, where he is also an adjunct instructor. Downloads/Updates


,concept,kind,score
0,software_development,utility,0.892096
1,technical_infrastructure,utility,0.748285
2,building_or_spatial_design,utility,0.676181
3,music_composition,utility,0.629438
4,live_music_or_recording,utility,0.604575
5,public_speaking_or_events,utility,0.573178
6,fitness_instruction,utility,0.559995
7,direct_gender_pronouns,direct_sensitive,0.517549


In [12]:
def show_high_calibrated_examples(concept, split_name="train", top_k=5):
    split_data = encoded_splits[split_name]
    concept_index = concept_names.index(concept)
    scores = split_data["calibrated_soft_targets"][:, concept_index].detach().cpu().numpy()
    top_indices = np.argsort(scores)[::-1][:top_k]
    return pd.DataFrame(
        {
            "score": scores[top_indices],
            "profession": [split_data["profession_labels"][idx] for idx in top_indices],
            "gender": [split_data["gender_labels"][idx] for idx in top_indices],
            "text": [split_data["texts"][idx] for idx in top_indices],
        }
    )


show_high_calibrated_examples("software_development")

,score,profession,gender,text
0,0.979394,architect,male,With two decades of engineering experience in ...
1,0.977525,software_engineer,male,Doug focuses on creating and improving DevOps ...
2,0.976033,architect,male,Oren spends lots of time developing open sourc...
3,0.975314,software_engineer,male,He has been involved in open source software f...
4,0.974595,software_engineer,female,She has been involved in all aspects of softwa...


In [13]:
batch_size = 256

def make_concept_qa_dataset(split_name):
    split_data = encoded_splits[split_name]
    return TensorDataset(
        split_data["embeddings"].cpu(),
        split_data["calibrated_soft_targets"].cpu(),
    )


train_dataset = make_concept_qa_dataset("train")
dev_dataset = make_concept_qa_dataset("dev")
test_dataset = make_concept_qa_dataset("test")

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

{
    "train_examples": len(train_dataset),
    "dev_examples": len(dev_dataset),
    "test_examples": len(test_dataset),
    "num_concepts": len(concept_names),
}

{'train_examples': 257478,
 'dev_examples': 39642,
 'test_examples': 99069,
 'num_concepts': 40}

In [14]:
def build_text_concept_qa_inputs(text_embeddings, concept_embeddings):
    batch_size = text_embeddings.size(0)
    num_concepts = concept_embeddings.size(0)
    text_rep = text_embeddings.unsqueeze(1).repeat(1, num_concepts, 1)
    concept_rep = concept_embeddings.unsqueeze(0).repeat(batch_size, 1, 1)
    return torch.cat([text_rep, concept_rep], dim=2).reshape(batch_size * num_concepts, -1)


def tensor_corrcoef(left, right):
    left = left.flatten()
    right = right.flatten()
    left = left - left.mean()
    right = right - right.mean()
    denom = left.norm() * right.norm()
    if float(denom) == 0.0:
        return torch.tensor(float("nan"), device=left.device)
    return (left @ right) / denom


def rank_tensor(values):
    order = values.flatten().argsort()
    ranks = torch.empty_like(order, dtype=torch.float32)
    ranks[order] = torch.arange(len(order), device=values.device, dtype=torch.float32)
    return ranks.view_as(values)


@torch.no_grad()
def evaluate_concept_qa_soft(model, loader):
    model.eval()
    losses = []
    hard_accuracies = []
    pred_batches = []
    target_batches = []

    for text_batch, target_batch in loader:
        text_batch = text_batch.to(device)
        target_batch = target_batch.to(device)
        inputs = build_text_concept_qa_inputs(text_batch, concept_embeddings)
        logits = model(inputs).view_as(target_batch)
        probs = torch.sigmoid(logits)
        loss = F.binary_cross_entropy_with_logits(logits, target_batch)
        preds = (probs >= 0.5).float()
        hard_targets = (target_batch >= 0.5).float()

        losses.append(float(loss.item()))
        hard_accuracies.append(float((preds == hard_targets).float().mean().item()))
        pred_batches.append(probs.detach().cpu())
        target_batches.append(target_batch.detach().cpu())

    all_preds = torch.cat(pred_batches)
    all_targets = torch.cat(target_batches)
    pearson = tensor_corrcoef(all_preds, all_targets)
    spearman = tensor_corrcoef(rank_tensor(all_preds), rank_tensor(all_targets))

    return {
        "loss": float(np.mean(losses)),
        "mae": float((all_preds - all_targets).abs().mean().item()),
        "rmse": float(torch.sqrt(((all_preds - all_targets) ** 2).mean()).item()),
        "pearson": float(pearson.item()),
        "spearman": float(spearman.item()),
        "hard_accuracy": float(np.mean(hard_accuracies)),
    }

In [15]:
concept_qa_model = ConceptNet2(embed_dims=embedding_dim).to(device)
optimizer = torch.optim.AdamW(concept_qa_model.parameters(), lr=1e-3, weight_decay=1e-4)
num_epochs = 3

concept_qa_history = []
for epoch in range(1, num_epochs + 1):
    concept_qa_model.train()
    train_losses = []

    for text_batch, target_batch in train_loader:
        text_batch = text_batch.to(device)
        target_batch = target_batch.to(device)

        inputs = build_text_concept_qa_inputs(text_batch, concept_embeddings)
        logits = concept_qa_model(inputs).view_as(target_batch)
        loss = F.binary_cross_entropy_with_logits(logits, target_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_losses.append(float(loss.item()))

    dev_metrics = evaluate_concept_qa_soft(concept_qa_model, dev_loader)
    concept_qa_history.append(
        {
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            "dev_loss": dev_metrics["loss"],
            "dev_mae": dev_metrics["mae"],
            "dev_rmse": dev_metrics["rmse"],
            "dev_pearson": dev_metrics["pearson"],
            "dev_spearman": dev_metrics["spearman"],
            "dev_hard_accuracy": dev_metrics["hard_accuracy"],
        }
    )

concept_qa_test_metrics = evaluate_concept_qa_soft(concept_qa_model, test_loader)

pd.DataFrame(concept_qa_history), concept_qa_test_metrics

(   epoch  train_loss  dev_loss   dev_mae  dev_rmse  dev_pearson  dev_spearman  \
 0      1    0.648020  0.646812  0.013213  0.016690     0.994167      0.993458   
 1      2    0.646929  0.647205  0.018029  0.021566     0.992385      0.991459   
 2      3    0.646832  0.646692  0.011831  0.015062     0.994970      0.994113   
 
    dev_hard_accuracy  
 0           0.964601  
 1           0.948777  
 2           0.967209  ,
 {'loss': 0.6468531181337913,
  'mae': 0.011816807091236115,
  'rmse': 0.015046825632452965,
  'pearson': 0.9950971007347107,
  'spearman': 0.9941980242729187,
  'hard_accuracy': 0.9671540081655025})

In [16]:
@torch.no_grad()
def predict_concept_scores_for_row(row_index, split_name="dev"):
    concept_qa_model.eval()
    split_data = encoded_splits[split_name]
    text_embedding = split_data["embeddings"][row_index : row_index + 1].to(device)
    inputs = build_text_concept_qa_inputs(text_embedding, concept_embeddings)
    logits = concept_qa_model(inputs).view(1, len(concept_names))
    return torch.sigmoid(logits).squeeze(0).cpu().numpy()


row_index = 0
split_name = "dev"
model_scores = predict_concept_scores_for_row(row_index, split_name=split_name)
target_scores = encoded_splits[split_name]["calibrated_soft_targets"][row_index].cpu().numpy()
top_indices = np.argsort(model_scores)[::-1][:8]

pd.DataFrame(
    {
        "concept": [concept_names[idx] for idx in top_indices],
        "kind": [concepts_df.loc[idx, "kind"] for idx in top_indices],
        "model_score": model_scores[top_indices],
        "target_score": target_scores[top_indices],
    }
)

,concept,kind,model_score,target_score
0,financial_or_accounting_work,utility,0.716145,0.695945
1,literary_writing,utility,0.653024,0.660420
2,building_or_spatial_design,utility,0.651115,0.663708
3,classroom_instruction,utility,0.629045,0.634557
4,appearance_or_body_presentation,proxy_sensitive,0.607142,0.612148
5,fashion_or_modeling_work,utility,0.542666,0.553160
6,technical_infrastructure,utility,0.539619,0.533686
7,software_development,utility,0.516272,0.500639


In [17]:
qa_experiment_name = "bias_in_bios_concept_qa_full"
qa_checkpoint = paths.checkpoints_root / f"concept_qa_{qa_experiment_name}.pt"
qa_history_path = paths.runs_root / f"concept_qa_{qa_experiment_name}_history.json"
qa_cache_path = paths.runs_root / f"concept_qa_{qa_experiment_name}_cache.pt"

cache_payload = {
    "encoder_name": encoder_name,
    "embedding_dim": embedding_dim,
    "concepts": concepts_df.to_dict(orient="records"),
    "concept_embeddings": concept_embeddings.detach().cpu(),
    "concept_dictionary": concept_dictionary.detach().cpu(),
    "calibration_medians": concept_medians.detach().cpu(),
    "calibration_stds": concept_stds.detach().cpu(),
    "calibration_temperature": calibration_temperature,
    "splits": {
        split_name: {
            "embeddings": split_data["embeddings"].detach().cpu(),
            "similarity_scores": split_data["similarity_scores"].detach().cpu(),
            "calibrated_soft_targets": split_data["calibrated_soft_targets"].detach().cpu(),
            "profession_ids": split_data["profession_ids"].cpu(),
            "gender_ids": split_data["gender_ids"].cpu(),
            "profession_labels": split_data["profession_labels"],
            "gender_labels": split_data["gender_labels"],
        }
        for split_name, split_data in encoded_splits.items()
    },
}

checkpoint_payload = {
    "model_state_dict": concept_qa_model.state_dict(),
    "embedding_dim": embedding_dim,
    "encoder_name": encoder_name,
    "concepts_path": str(concepts_path.relative_to(repo_root)),
    "qa_cache_path": str(qa_cache_path.relative_to(repo_root)),
    "calibration_temperature": calibration_temperature,
    "test_metrics": concept_qa_test_metrics,
}

torch.save(cache_payload, qa_cache_path)
torch.save(checkpoint_payload, qa_checkpoint)

history_payload = {
    "history": concept_qa_history,
    "test_metrics": concept_qa_test_metrics,
}
with open(qa_history_path, "w", encoding="utf-8") as handle:
    json.dump(history_payload, handle, indent=2)

{
    "checkpoint": str(qa_checkpoint.relative_to(repo_root)),
    "history": str(qa_history_path.relative_to(repo_root)),
    "cache": str(qa_cache_path.relative_to(repo_root)),
}

{'checkpoint': 'artifacts/models/concept_qa_bias_in_bios_concept_qa_full.pt',
 'history': 'artifacts/runs/concept_qa_bias_in_bios_concept_qa_full_history.json',
 'cache': 'artifacts/runs/concept_qa_bias_in_bios_concept_qa_full_cache.pt'}